In [6]:
import pandas as pd
import numpy as np
import sys
import os
import librosa
from scipy.signal import savgol_filter

sys.path.append(os.path.abspath(".."))  
sys.path.append(os.path.abspath("./src")) 

from src.data_processing import custom_spectrogram, FCC

# Load dataset file with labels and features (FCC)
data_path = "../data/"
audio_path = "../audios/"


#dataset ='Dataset_1/'
#dataset ='Dataset_2/'
dataset ='Automatic_Labeling_Kyoogu/'

file = 'Labeled_Data.xlsx'
G_matrix_segments = pd.read_excel(data_path+dataset+file)
G_matrix_segments

,File,Specie ID,Start,End,FminVoc,FmaxVoc
0,SMA03126_20210611_200000.wav,Boana_platanera,1.2,3.199,200,2500
1,SMA03126_20210611_200000.wav,Boana_platanera,2.0,3.999,200,2500
2,SMA03126_20210611_200000.wav,Boana_platanera,8.8,10.799,200,2500
3,SMA03126_20210611_200000.wav,Boana_platanera,9.6,11.599,200,2500
4,SMA03126_20210611_200000.wav,Boana_platanera,12.8,14.799,200,2500
...,...,...,...,...,...,...
1377,SMA03251_20210605_053000.wav,Troglodytes_aedon,54.0,55.999,400,7000
1378,SMA03251_20210608_073000.wav,Troglodytes_aedon,10.8,12.799,400,7000
1379,SMA03251_20210608_073000.wav,Troglodytes_aedon,32.8,34.799,400,7000
1380,SMA03251_20210608_073000.wav,Troglodytes_aedon,43.6,45.599,400,7000


In [7]:
if dataset == 'Automatic_Labeling_Kyoogu/':
    dataset = 'Dataset_1/'

# Parameters
window_size = 1024
hop_size = 512
nfft = 2048

# Dictionary to store spectrograms of each audio file of the dataset
spectrogram_dict = {}

# Process audio files
for file in G_matrix_segments['File'].unique():
    audio_path_dataset = audio_path+dataset+file
    signal, sr = librosa.load(audio_path_dataset, sr=None, mono=False)
    S, f, t = custom_spectrogram(signal, sr, window_size, hop_size, nfft)
    spectrogram_dict[file] = {"spectrogram": S, "frequencies": f, "times": t}

In [ ]:
# Calculate features

Y_spectral_features = []
fcc_features_list = []

for index, row in G_matrix_segments.iterrows():
    file_name = row['File']
    start_time = row['Start']
    end_time = row['End']
    fmin_voc = row['FminVoc']
    fmax_voc = row['FmaxVoc']
    specie = row['Specie ID']
    
    features = {}    
    
    if file_name in spectrogram_dict:
        Sxx = spectrogram_dict[file_name]['spectrogram']
        f = spectrogram_dict[file_name]['frequencies']
        t = spectrogram_dict[file_name]['times']

        # Flip spectrogram and frequency axis
        Sxx_flip = np.flip(Sxx, axis=0)
        f1 = np.flip(f)

        # Find the corresponding indices in the spectrogram
        posX = np.argmin(np.abs(t - start_time))
        posXplusW = np.argmin(np.abs(t - end_time))
        posY = np.argmin(np.abs(f1 - fmin_voc))
        posYplusH = np.argmin(np.abs(f1 - fmax_voc))

        if posX == 0:
            posX =1

        # Extract the spectrogram segment
        H_Segment = Sxx_flip[posYplusH:posY+1, posX-1:posXplusW]

        # Compute the dominant frequency
        sum_domin = np.sum(H_Segment, axis=1)
        sum_domin_smooth = savgol_filter(sum_domin, window_length=11, polyorder=2)
        dom_idx = np.argmax(sum_domin_smooth)
        dominant_frequency = ((((fmin_voc * Sxx.shape[0] / (f[-1])) + dom_idx) / Sxx.shape[0]) * f[-1])

        # Calculate FCCs
        nfrec = 4
        div = 4
        nfiltros = 14
        Fccs = FCC(H_Segment, nfiltros, nfrec, div)

        # Add differential FCCs (1st and 2nd order differences)
        dfcc = np.diff(Fccs, n=1, axis=1)
        dfcc2 = np.diff(Fccs, n=2, axis=1)

        Fccs_flat = np.concatenate([
            Fccs.flatten(order='F'),
            np.mean(dfcc, axis=1),
            np.mean(dfcc2, axis=1)
        ])

        # Store the feature vector
        fcc_features_list.append(Fccs_flat)

        # Compute the dominant frequency
        sum_domin = np.sum(H_Segment, axis=1)  
        sum_domin_smooth = savgol_filter(sum_domin, window_length=11, polyorder=2)  
        dom_idx = np.argmax(sum_domin_smooth)  
        dominant_frequency2 = ((((fmin_voc * Sxx.shape[0]  / (sr / 2)) + dom_idx) / Sxx.shape[0] ) * sr / 2)

        # Compute spectral centroid
        spectral_centroid = np.sum(H_Segment * f1[posYplusH:posY+1, None], axis=0) / np.sum(H_Segment, axis=0)
        spectral_centroid = np.nanmean(spectral_centroid)

        # Compute spectral bandwidth
        spectral_bandwidth = np.sqrt(np.sum(((f1[posYplusH:posY+1, None] - spectral_centroid) ** 2) * H_Segment, axis=0) / np.sum(H_Segment, axis=0))
        spectral_bandwidth = np.nanmean(spectral_bandwidth)
        
        # Compute spectral flatness
        spectral_flatness = np.exp(np.mean(np.log(H_Segment + 1e-10), axis=0)) / np.mean(H_Segment, axis=0)
        spectral_flatness = np.nanmean(spectral_flatness)   

        features={
            'File': file_name,
            'Specie ID': specie,
            'Start':start_time,
            'End': end_time,
            'Length':end_time -start_time,
            'Fdom': dominant_frequency,
            'FminVoc':  fmin_voc,
            'FmaxVoc': fmax_voc
        }

        # Add FCCs to features
        for idx, value in enumerate(Fccs_flat[1:], start=1):
            features[f'FCC{idx}'] = value
    Y_spectral_features.append(features)

    # Add remaining features after FCCs
    features['SpectralCentroid'] = spectral_centroid
    features['Bandwidth'] = spectral_bandwidth
    features['SpectralFlatness'] = spectral_flatness
    features['DeltaFreq'] = (fmax_voc - fmin_voc)

Y_df_spectral_features = pd.DataFrame(Y_spectral_features)

df_filtered = Y_df_spectral_features[Y_df_spectral_features['Specie ID'] != 'noise']
df_filtered = df_filtered.dropna(subset=['Specie ID'])
print(df_filtered['Specie ID'].value_counts())  

Specie ID
Boana_platanera                349
Leptodactylus_fragilis         325
Leptodactylus_fuscus           305
Patagioenas_cayennensis        165
Dendropsophus_microcephalus    112
Dendroplex_picus                76
Troglodytes_aedon               29
Alouatta_sp                     21
Name: count, dtype: int64


In [9]:
df_filtered

,File,Specie ID,Start,End,Length,Fdom,FminVoc,FmaxVoc,FCC1,FCC2,...,FCC18,FCC19,FCC20,FCC21,FCC22,FCC23,SpectralCentroid,Bandwidth,SpectralFlatness,DeltaFreq
0,SMA03126_20210611_200000.wav,Boana_platanera,1.2,3.199,1.999,2494.634146,200,2500,-2.528486,2.471312,...,-0.024349,-0.082509,0.847230,0.542713,-0.014309,0.477499,996.273655,723.668685,0.664043,2300
1,SMA03126_20210611_200000.wav,Boana_platanera,2.0,3.999,1.999,2494.634146,200,2500,-3.423150,1.797186,...,0.101820,0.143538,0.480819,-0.273335,-0.601963,-0.219395,991.029722,714.181321,0.663596,2300
2,SMA03126_20210611_200000.wav,Boana_platanera,8.8,10.799,1.999,2494.634146,200,2500,-1.643535,2.881541,...,-0.110768,-0.020601,1.004440,0.444135,0.034643,0.260648,1109.374423,754.315252,0.687174,2300
3,SMA03126_20210611_200000.wav,Boana_platanera,9.6,11.599,1.999,2494.634146,200,2500,-1.397782,2.849785,...,-0.485911,-0.380565,0.651914,-0.221829,-0.318634,0.077291,1073.497766,736.898649,0.681053,2300
4,SMA03126_20210611_200000.wav,Boana_platanera,12.8,14.799,1.999,2494.634146,200,2500,-2.245177,1.310356,...,0.262433,0.037083,-0.557672,-1.114157,-1.288006,-0.689676,1092.794373,746.600309,0.681270,2300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,SMA03251_20210605_053000.wav,Troglodytes_aedon,54.0,55.999,1.999,400.000000,400,7000,1.983473,0.131651,...,-0.673737,0.361401,-7.021199,-2.407979,1.300261,0.251153,4585.321267,1883.102118,0.559634,6600
1378,SMA03251_20210608_073000.wav,Troglodytes_aedon,10.8,12.799,1.999,400.000000,400,7000,3.066072,-1.459103,...,-0.050859,0.281079,3.286226,1.446850,-0.891702,-1.679020,4456.672917,1996.267191,0.644956,6600
1379,SMA03251_20210608_073000.wav,Troglodytes_aedon,32.8,34.799,1.999,4286.829268,400,7000,2.818356,-1.296832,...,-1.210085,1.175252,-6.574594,-2.944687,0.266437,1.591550,4255.346311,1752.463462,0.558093,6600
1380,SMA03251_20210608_073000.wav,Troglodytes_aedon,43.6,45.599,1.999,470.243902,400,7000,0.606844,2.580560,...,-2.293107,0.316643,-3.292698,-1.012381,1.185086,1.497484,4529.220736,1933.704949,0.555225,6600


In [10]:
Y_df_spectral_features

,File,Specie ID,Start,End,Length,Fdom,FminVoc,FmaxVoc,FCC1,FCC2,...,FCC18,FCC19,FCC20,FCC21,FCC22,FCC23,SpectralCentroid,Bandwidth,SpectralFlatness,DeltaFreq
0,SMA03126_20210611_200000.wav,Boana_platanera,1.2,3.199,1.999,2494.634146,200,2500,-2.528486,2.471312,...,-0.024349,-0.082509,0.847230,0.542713,-0.014309,0.477499,996.273655,723.668685,0.664043,2300
1,SMA03126_20210611_200000.wav,Boana_platanera,2.0,3.999,1.999,2494.634146,200,2500,-3.423150,1.797186,...,0.101820,0.143538,0.480819,-0.273335,-0.601963,-0.219395,991.029722,714.181321,0.663596,2300
2,SMA03126_20210611_200000.wav,Boana_platanera,8.8,10.799,1.999,2494.634146,200,2500,-1.643535,2.881541,...,-0.110768,-0.020601,1.004440,0.444135,0.034643,0.260648,1109.374423,754.315252,0.687174,2300
3,SMA03126_20210611_200000.wav,Boana_platanera,9.6,11.599,1.999,2494.634146,200,2500,-1.397782,2.849785,...,-0.485911,-0.380565,0.651914,-0.221829,-0.318634,0.077291,1073.497766,736.898649,0.681053,2300
4,SMA03126_20210611_200000.wav,Boana_platanera,12.8,14.799,1.999,2494.634146,200,2500,-2.245177,1.310356,...,0.262433,0.037083,-0.557672,-1.114157,-1.288006,-0.689676,1092.794373,746.600309,0.681270,2300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,SMA03251_20210605_053000.wav,Troglodytes_aedon,54.0,55.999,1.999,400.000000,400,7000,1.983473,0.131651,...,-0.673737,0.361401,-7.021199,-2.407979,1.300261,0.251153,4585.321267,1883.102118,0.559634,6600
1378,SMA03251_20210608_073000.wav,Troglodytes_aedon,10.8,12.799,1.999,400.000000,400,7000,3.066072,-1.459103,...,-0.050859,0.281079,3.286226,1.446850,-0.891702,-1.679020,4456.672917,1996.267191,0.644956,6600
1379,SMA03251_20210608_073000.wav,Troglodytes_aedon,32.8,34.799,1.999,4286.829268,400,7000,2.818356,-1.296832,...,-1.210085,1.175252,-6.574594,-2.944687,0.266437,1.591550,4255.346311,1752.463462,0.558093,6600
1380,SMA03251_20210608_073000.wav,Troglodytes_aedon,43.6,45.599,1.999,470.243902,400,7000,0.606844,2.580560,...,-2.293107,0.316643,-3.292698,-1.012381,1.185086,1.497484,4529.220736,1933.704949,0.555225,6600
